# GPU inference service: concurrency-4 check
The concurrency-1 GPU smoke check passed. This next check submits up to four HTTP requests concurrently to one serialized model worker. It tests queueing and output correctness, not GPU batching.

Select NVIDIA T4, then Run all. Defaults: 30 requests per configuration, 5 configurations, 150 request executions, float32. The result ZIP downloads automatically. No source upload or Google Drive is required.

Request processing order can differ between configurations, so cache-hit differences here are not controlled policy comparisons. Instrumented timings are not production performance claims.


In [ ]:
import os, sys, subprocess, hashlib, json
from pathlib import Path
from google.colab import files
repo = Path("/content/kv-serving-source")
url = "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, str(repo)], check=True)
os.chdir(repo)
checksums = {str(p.relative_to(repo)): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in (repo / "src").rglob("*.py")}
(repo / "source-manifest.json").write_text(json.dumps(checksums, indent=2))
os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[serving]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select GPU runtime'; print(torch.cuda.get_device_name())"], check=True)
subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([python, "-m", "kv_cache.cli", "experiment", "--output", "results/experiment"], check=True)


## Run concurrency validation
Keep the defaults. Each configuration starts a fresh service. Any output mismatch stops the suite and preserves diagnostics. Download the archive even if a check fails.


In [ ]:
CONCURRENCY = 4
WORKLOAD = "hot"
SEED = 99
REQUESTS = 30
REPEATS = 1
BUDGET_MIB = 32
from datetime import datetime, timezone
run_dir = Path("/content") / ("kv-serving-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
command = [python, "-u", "-m", "kv_cache.serve_eval", "--predictor", "results/experiment/predictor.json",
           "--device", "cuda", "--requests", str(REQUESTS), "--repeats", str(REPEATS),
           "--seed", str(SEED), "--workload", WORKLOAD, "--budget-mib", str(BUDGET_MIB),
           "--concurrency", str(CONCURRENCY), "--output", str(run_dir)]
try:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Evaluation failed; download diagnostics below")
finally:
    if run_dir.exists():
        import shutil
        shutil.copyfile(repo / "source-manifest.json", run_dir / "source-manifest.json")
        shutil.make_archive(str(run_dir), "zip", run_dir)
    if Path(archive).exists():
        files.download(archive)


In [ ]:
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Status:", manifest["status"])
for row in manifest["completed"]:
    report = json.loads((run_dir / (row["label"] + ".json")).read_text())
    order = [r["sequence"] for r in report["requests"]]
    print(row["label"], row["correctness"]["status"],
          "HTTP p95 ms:", round(report["http_p95_ms"], 2),
          "queue p95 ms:", round(report["queue_p95_ms"], 2),
          "requests/s:", round(report["requests_per_second"], 2),
          "input order preserved:", order == list(range(len(order))))
print("Concurrency means queued HTTP requests, not parallel GPU generation.")


## Download again if needed
This cell only downloads the most recent ZIP. It does not rerun the experiment. If the runtime was reset, an undownloaded archive may no longer exist.


In [ ]:
from pathlib import Path
from google.colab import files
archives = sorted(Path("/content").glob("kv-serving-*.zip"))
if archives:
    files.download(str(archives[-1]))
else:
    print("No serving archive found in this runtime.")
